In [1]:
import pandas as pd

from typing import Any, List, Tuple

from dataclasses import dataclass


from datetime import datetime

from utils.types import date

In [ ]:
df = pd.read_excel(
    "./../../upgrade_mandi/raw-sheets-dump/b2b/Transaction List.xlsx",
    sheet_name="Sheet1",
)
df.columns = [column.strip().title() for column in df.columns]

In [5]:
df = (
    df[
        [
            "Date",
            "Name",
            "Address",
            "Item Name",
            "Qty",
            "Rate",
        ]
    ]
    .dropna(
        subset=[
            "Item Name",
            "Qty",
            "Rate",
        ],
        how="all",
    )
    .reset_index(drop=True)
)

In [6]:
def lower(row: pd.Series) -> pd.Series:
    for key in row.keys():
        row[key] = str(row[key]).title()
    return row


df[["Address", "Name", "Item Name"]] = df[["Address", "Name", "Item Name"]].apply(lower)
df[["Qty", "Rate"]] = df[["Qty", "Rate"]].astype(int)

df["Date"] = df["Date"].apply(lambda date: date.strftime("%d-%m-%Y"))

In [7]:
df

,Date,Name,Address,Item Name,Qty,Rate
0,14-09-2025,Manoj Jaiswal,Sadar,Retail G4 Potato,100,23
1,14-09-2025,Manoj Jaiswal,Sadar,Retail Onion,100,18
2,14-09-2025,Jagdish Shahu,Besa,Retail Agra Potato,50,21
3,14-09-2025,Jagdish Shahu,Besa,Retail Onion,50,18
4,14-09-2025,Mangesh Mankar,Jaitala,Retail Agra Potato,50,21
...,...,...,...,...,...,...
147,21-09-2025,Bhiwa Kawle,Laxmi Nagar,Retail G4 Potato,50,21
148,21-09-2025,Bhiwa Kawle,Laxmi Nagar,Regular Garlic,4,105
149,21-09-2025,Omprakash Shahu,Hingna,Wholsale Potato,50,17
150,21-09-2025,Shagoon Vegetables,Hingna,B Onion,100,10


In [ ]:
customer_names = df["Name"].unique()
dates = df["Date"].unique()
items = df["Item Name"].unique()

groups = df.groupby(["Date", "Name", "Address"]).groups

newDf = df.drop(labels=["Date", "Name", "Address"], axis=1).copy()

# groups = { key: df.iloc[value] for key, value in groups.items() }


@dataclass
class Customer:
    customer_name: str
    location: str
    data: pd.DataFrame

    def __repr__(self):
        return f"{self.customer_name} - {self.location}"


by_dates: dict[str, List[Customer]] = {}

for _key, value in groups.items():
    key: Tuple[str, str, str] = tuple(_key)  # type: ignore
    if key[0] not in by_dates:
        by_dates[key[0]] = []
    customer = Customer(key[1], key[2], newDf.iloc[value])
    by_dates[key[0]].append(customer)

by_dates

{'14-09-2025': [Jagdish Shahu - Besa,
  Mangesh Mankar - Jaitala,
  Manoj Jaiswal - Sadar,
  Omprakash Shahu - Hingna,
  Shagoon Vegetables - Hingna,
  Shubham Misal - Dighori,
  Swapnil Tratak - Mhalgi Nagar,
  Uttam Shahu - Besa],
 '15-09-2025': [Amit Shahu - Spurti Bazar Beltarodi,
  Jagdish Shahu - Besa,
  Manoj Jaiswal - Sadar,
  Omprakash Shahu - Hingna,
  Shagoon Vegetables - Hingna,
  Shubham Misal - Dighori,
  Swapnil Tratak - Mhalgi Nagar,
  Uttam Shahu - Besa,
  Vishal Jaiswal - Dighori],
 '16-09-2025': [Amit Shahu - Spurti Bazar Beltarodi,
  Jagdish Shahu - Besa,
  Krishna Itwari - Itwari,
  Shagoon Vegetables - Hingna,
  Shubham Misal - Dighori,
  Swapnil Tratak - Mhalgi Nagar,
  Uttam Shahu - Besa,
  Vishal Jaiswal - Dighori],
 '17-09-2025': [Amit Shahu - Spurti Bazar Beltarodi,
  Bhiwa Kawle - Laxmi Nagar,
  Jagdish Shahu - Besa,
  Manoj Jaiswal - Sadar,
  Sudhir Dharmashali - Kahmla,
  Swapnil Tratak - Mhalgi Nagar],
 '18-09-2025': [Amit Shahu - Spurti Bazar Beltarodi,
